# ACTIVIDAD 20
VICTOR CEN


# El problema del viajero (TSP)

El problema del viajero de comercio, conocido como **TSP** (*Traveling Salesman Problem*), es un problema clásico de optimización combinatoria. Consiste en encontrar la ruta más corta posible para que un viajero recorra un conjunto de ciudades, visite cada una exactamente una vez y finalmente regrese a la ciudad de origen.

Este problema aparece en aplicaciones reales como rutas de reparto, planeación logística, manufactura, redes, robótica y secuenciación de procesos.

## Descripción general

El TSP puede representarse mediante un conjunto de ciudades y una matriz de distancias, donde cada elemento indica el costo o distancia de viajar de una ciudad a otra.

El objetivo es determinar el orden óptimo de visita de las ciudades. Aunque el planteamiento parece sencillo, el número de rutas posibles crece muy rápidamente conforme aumenta el número de ciudades. Por ello, el TSP es considerado un problema difícil desde el punto de vista computacional.

En este notebook se utiliza el **algoritmo de colonia de hormigas** para buscar una solución buena o cercana a la óptima. Este método está inspirado en el comportamiento de las hormigas reales, las cuales dejan rastros de feromonas al desplazarse. Con el tiempo, los caminos más convenientes acumulan más feromona y se vuelven más atractivos para otras hormigas.

## Objetivos

Los principales objetivos al estudiar el problema del viajero son los siguientes:

- Minimizar la distancia total recorrida.
- Reducir costos asociados al transporte, combustible o tiempo.
- Analizar técnicas de optimización aplicables a problemas complejos.
- Comparar métodos exactos y heurísticos.
- Implementar una metaheurística inspirada en la naturaleza, como la colonia de hormigas, para obtener soluciones eficientes en tiempos razonables.

## Limitaciones

A pesar de su importancia, el TSP presenta varias limitaciones y dificultades:

- El número de rutas posibles crece de forma factorial con el número de ciudades.
- Encontrar la solución exacta puede ser inviable para instancias grandes.
- Muchas formulaciones suponen distancias fijas, cuando en la realidad pueden cambiar por tráfico, clima o restricciones.
- El modelo clásico considera un solo viajero y una sola visita por ciudad.
- No contempla, en su versión básica, ventanas de tiempo, capacidades, prioridades ni restricciones logísticas adicionales.

Por estas razones, en muchos casos se prefieren algoritmos heurísticos o metaheurísticos que encuentren soluciones muy buenas sin garantizar necesariamente la óptima exacta.

In [7]:
import random, sys, math

# Nota: en lugar de matrices se usan listas de listas

# Genera una matriz de distancias de nCiudades x nCiudades
def matrizDistancias(nCiud, distanciaMaxima):
    matriz = [[0 for i in range(nCiud)] for j in range(nCiud)]

    for i in range(nCiud):
        for j in range(i):
            matriz[i][j] = distanciaMaxima * random.random()
            matriz[j][i] = matriz[i][j]

    return matriz

# Elige un paso de una hormiga, teniendo en cuenta las distancias
# y las feromonas y descartando las ciudades ya visitadas.
def eligeCiudad(dists, ferom, visitadas):
    # Se calcula la tabla de pesos de cada ciudad
    listaPesos = []
    disponibles = []
    actual = visitadas[-1]

    # Influencia de cada valor (alfa: feromonas; beta: distancias)
    alfa = 1.0
    beta = 0.5

    # El parámetro beta (peso de las distancias) es 0.5, alfa=1.0
    for i in range(len(dists)):
        if i not in visitadas:
            fer = math.pow((1.0 + ferom[actual][i]), alfa)
            peso = math.pow(1.0 / dists[actual][i], beta) * fer
            disponibles.append(i)
            listaPesos.append(peso)

    # Se elige aleatoriamente una de las ciudades disponibles,
    # teniendo en cuenta su peso relativo.
    valor = random.random() * sum(listaPesos)
    acumulado = 0.0
    i = -1
    while valor > acumulado:
        i += 1
        acumulado += listaPesos[i]

    return disponibles[i]

# Genera una "hormiga", que elegirá un camino teniendo en cuenta
# las distancias y los rastros de feromonas. Devuelve una tupla
# con el camino y su longitud.
def eligeCamino(distancias, feromonas):
    # La ciudad inicial siempre es la 0
    camino = [0]
    longCamino = 0

    # Elegir cada paso según la distancia y las feromonas
    while len(camino) < len(distancias):
        ciudad = eligeCiudad(distancias, feromonas, camino)
        longCamino += distancias[camino[-1]][ciudad]
        camino.append(ciudad)

    # Para terminar hay que volver a la ciudad de origen (0)
    longCamino += distancias[camino[-1]][0]
    camino.append(0)

    return (camino, longCamino)

# Actualiza la matriz de feromonas siguiendo el camino recibido
def rastroFeromonas(feromonas, camino, dosis):
    for i in range(len(camino) - 1):
        feromonas[camino[i]][camino[i + 1]] += dosis

# Evapora todas las feromonas multiplicándolas por una constante
# = 0.9 (en otras palabras, el coeficiente de evaporación es 0.1)
def evaporaFeromonas(feromonas):
    for lista in feromonas:
        for i in range(len(lista)):
            lista[i] *= 0.9

# Resuelve el problema del viajante de comercio mediante el
# algoritmo de la colonia de hormigas. Recibe una matriz de
# distancias y devuelve una tupla con el mejor camino que ha
# obtenido (lista de índices) y su longitud
def hormigas(distancias, iteraciones, distMedia):
    # Primero se crea una matriz de feromonas vacía
    n = len(distancias)
    feromonas = [[0 for i in range(n)] for j in range(n)]

    # El mejor camino y su longitud (inicialmente "infinita")
    mejorCamino = []
    longMejorCamino = sys.maxsize

    # En cada iteración se genera una hormiga, que elige un camino,
    # y si es mejor que el mejor que teníamos, deja su rastro de
    # feromonas (mayor cuanto más corto sea el camino)
    for iter in range(iteraciones):
        (camino, longCamino) = eligeCamino(distancias, feromonas)

        if longCamino <= longMejorCamino:
            mejorCamino = camino
            longMejorCamino = longCamino

            rastroFeromonas(feromonas, camino, distMedia / longCamino)

        # En cualquier caso, las feromonas se van evaporando
        evaporaFeromonas(feromonas)

    # Se devuelve el mejor camino que se haya encontrado
    return (mejorCamino, longMejorCamino)

# Generación de una matriz de prueba
numCiudades = 10
distanciaMaxima = 10
ciudades = matrizDistancias(numCiudades, distanciaMaxima)

# Obtención del mejor camino
iteraciones = 1000
distMedia = numCiudades * distanciaMaxima / 2
(camino, longCamino) = hormigas(ciudades, iteraciones, distMedia)
print("Camino:", camino)
print("Longitud del camino:", longCamino)

Camino: [0, 9, 5, 3, 7, 4, 8, 1, 2, 6, 0]
Longitud del camino: 15.223021267072495
